# SolarGuard — Cloud Baseline Training (Google Colab)

Runs the **already-approved** SolarGuard baseline CNN experiment on a Colab GPU. This notebook
does not redesign anything — same dataset, same 540/117/115 split (seed 42), same architecture,
same loss, same augmentation policy as the local runs already recorded in `PLANNING.md`. Its only
job is to supply GPU compute this laptop doesn't have enough RAM/VRAM for.

**Dataset license: CC BY-NC-SA 4.0 (non-commercial)** — source: [kaggle.com/datasets/alicjalena/pv-panel-defect-dataset](https://www.kaggle.com/datasets/alicjalena/pv-panel-defect-dataset). See `DATASET.md` for full provenance notes.

**This notebook does not load or evaluate the test set at any point.**

In [ ]:
import os, subprocess, tempfile
from pathlib import Path

# 'x-access-token' is a non-secret placeholder username. Git persists the remote URL
# verbatim into .git/config, so the real PAT must never appear here — it is supplied
# through GIT_ASKPASS below instead.
REPO_URL = "https://x-access-token@github.com/dayush685-debug/solarguard.git"
CLONE_DIR = "/content/Solar_Guard"

# Prefer Colab Secrets (key icon in the left sidebar -> add GITHUB_TOKEN). Falls back to a
# hidden prompt so the notebook still works without them.
token = None
try:
    from google.colab import userdata
    token = userdata.get("GITHUB_TOKEN")
except Exception:
    pass
if not token:
    from getpass import getpass
    token = getpass("GitHub PAT (hidden; not written to the notebook or .git/config): ")

# Transient askpass helper. Note the script itself holds no secret — it only reads an
# environment variable that exists for the lifetime of these subprocess calls.
_askpass_dir = Path(tempfile.mkdtemp())
_askpass = _askpass_dir / "askpass.sh"
_askpass.write_text('#!/bin/sh\necho "$GH_TOKEN"\n')
_askpass.chmod(0o700)
_env = {**os.environ, "GH_TOKEN": token, "GIT_ASKPASS": str(_askpass), "GIT_TERMINAL_PROMPT": "0"}

try:
    if not os.path.exists(CLONE_DIR):
        subprocess.run(["git", "clone", REPO_URL, CLONE_DIR], env=_env, check=True)
    else:
        # Directory already present. Guarding only on existence silently re-used a stale
        # checkout and hid an already-pushed fix, so bring it CURRENT instead of trusting it.
        # set-url first: a clone made by an earlier version of this notebook may still carry
        # a PAT in its .git/config, and this overwrites it.
        subprocess.run(["git", "-C", CLONE_DIR, "remote", "set-url", "origin", REPO_URL], env=_env, check=True)
        subprocess.run(["git", "-C", CLONE_DIR, "fetch", "origin", "--quiet"], env=_env, check=True)
        subprocess.run(["git", "-C", CLONE_DIR, "reset", "--hard", "origin/main"], env=_env, check=True)
finally:
    _askpass.unlink(missing_ok=True)
    _askpass_dir.rmdir()
    del token, _env

%cd /content/Solar_Guard
REPO_ROOT = Path.cwd()
print(f"REPO_ROOT = {REPO_ROOT}")

print("\n--- checked-out commit (a stale clone can no longer hide) ---")
!git log -1 --oneline
print("\n--- stored remote URL (must contain NO token) ---")
!git remote -v

In [ ]:
!pip install -e . --quiet

In [ ]:
import sys, torch, torchvision
print(f"Python:      {sys.version.split()[0]}")
print(f"PyTorch:     {torch.__version__}")
print(f"torchvision: {torchvision.__version__}")
print(f"CUDA avail:  {torch.cuda.is_available()}")

if not torch.cuda.is_available():
    raise RuntimeError(
        "CUDA is not available in this Colab session \u2014 stopping per SolarGuard policy "
        "(the entire point of this notebook is GPU compute). "
        "Go to Runtime > Change runtime type > select a GPU, then re-run."
    )

print(f"CUDA version: {torch.version.cuda}")
print(f"GPU:          {torch.cuda.get_device_name(0)}")
print(f"VRAM:         {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
device = torch.device("cuda")

## Dataset acquisition

Downloaded fresh from Kaggle each session (not re-hosted anywhere) — matches the source
documented in `DATASET.md`. Requires a free Kaggle account and API token (`kaggle.json`),
uploaded below and **never committed to the repository**.

License: **CC BY-NC-SA 4.0 — non-commercial use only.**

In [ ]:
DATA_ROOT = REPO_ROOT / "data" / "candidates" / "PV_Panel_Defect_Dataset"

if not DATA_ROOT.exists():
    from google.colab import files
    print("Upload your kaggle.json API token (never committed to the repo):")
    files.upload()
    !mkdir -p ~/.kaggle && cp kaggle.json ~/.kaggle/ && chmod 600 ~/.kaggle/kaggle.json
    !pip install kaggle --quiet
    !kaggle datasets download -d alicjalena/pv-panel-defect-dataset -p /content/_raw_download
    DATA_ROOT.mkdir(parents=True, exist_ok=True)
    !unzip -q /content/_raw_download/*.zip -d {DATA_ROOT}
    print(f"dataset extracted to {DATA_ROOT}")
else:
    print(f"dataset already present at {DATA_ROOT}, skipping download")

In [ ]:
import pandas as pd, json

SPLITS_DIR = REPO_ROOT / "data" / "splits"
manifest = pd.read_csv(SPLITS_DIR / "manifest.csv")
class_to_idx = json.loads((SPLITS_DIR / "class_mapping.json").read_text())

assert len(manifest) == 772, f"expected 772 verified-unique images, got {len(manifest)}"
counts = manifest["split"].value_counts()
assert counts["Train"] == 540 and counts["Valid"] == 117 and counts["Test"] == 115

missing = [p for p in manifest["path"] if not (DATA_ROOT / p).exists()]
assert not missing, f"{len(missing)} manifest files missing after download, e.g. {missing[:5]}"

test_row_count = counts["Test"]  # read for verification ONLY \u2014 no Dataset/DataLoader built from it
print("dataset verified: 772 images, 540/117/115 split, all files present on disk")
print(f"test split confirmed present ({test_row_count} rows) but NOT loaded into any DataLoader")

In [ ]:
!python -m pytest tests/ -v

In [ ]:
from solarguard.models.baseline_cnn import BaselineCNN, count_parameters
from solarguard.data.datasets import build_train_val_dataloaders
from solarguard.training.train import set_seed

set_seed(42)
model = BaselineCNN(num_classes=6).to(device)
print(f"model constructed: {count_parameters(model):,} parameters")

train_loader, val_loader = build_train_val_dataloaders(
    SPLITS_DIR, DATA_ROOT, REPO_ROOT / "configs" / "preprocessing.yaml",
    batch_size=32, num_workers=0, seed=42,  # num_workers=0 kept, as instructed
)
images, labels = next(iter(train_loader))
images, labels = images.to(device), labels.to(device)
logits = model(images)
assert logits.shape == (images.size(0), 6)
print(f"forward pass OK: {tuple(images.shape)} -> {tuple(logits.shape)}")

In [ ]:
from solarguard.training.losses import build_loss, class_weights_from_train_split

class_weights = class_weights_from_train_split(SPLITS_DIR).to(device)
loss_fn = build_loss(class_weights)
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)

before = [p.clone() for p in model.parameters()]
optimizer.zero_grad()
loss = loss_fn(logits, labels)
loss.backward()
optimizer.step()
changed = any(not torch.equal(b, a) for b, a in zip(before, model.parameters()))

print(f"loss: {loss.item():.4f}")
print(f"weights changed after backward+step: {changed}")
assert changed, "SMOKE TEST FAILED"
print()
print("SMOKE TEST PASSED. Full training NOT started \u2014 see next cell.")

## STOP HERE

Everything above is verification only. The full-training cell below is intentionally gated
and must be enabled deliberately (`RUN_FULL_TRAINING = True`) \u2014 it does not run via
"Run All" by accident.

In [ ]:
RUN_FULL_TRAINING = False  # must be manually flipped to True \u2014 deliberate friction, not an accident waiting to happen

if RUN_FULL_TRAINING:
    from datetime import datetime
    from solarguard.training.config import TrainingConfig
    from solarguard.training.train import fit

    run_id = datetime.now().strftime("colab_run_%Y%m%d_%H%M%S")
    experiment_dir = REPO_ROOT / "experiments" / "baseline_cnn" / run_id
    config = TrainingConfig(experiment_dir=experiment_dir, num_workers=0)  # identical to the approved local config

    fresh_model = BaselineCNN(num_classes=config.num_classes).to(device)
    result = fit(fresh_model, train_loader, val_loader, loss_fn, config, list(class_to_idx))
    print(result)
else:
    print("RUN_FULL_TRAINING is False \u2014 full training not started, as instructed.")

In [ ]:
if RUN_FULL_TRAINING:
    import shutil
    # history.csv / metrics.json / config.yaml / curves / confusion matrix saved the same way
    # run_baseline_experiment.py already does locally \u2014 same format, same filenames, so
    # local and Colab experiment folders are directly comparable
    zip_path = shutil.make_archive(str(experiment_dir), "zip", experiment_dir)
    from google.colab import files
    files.download(zip_path)
    print(f"downloaded: {zip_path}")

In [ ]:
# ---------------------------------------------------------------------------
# Post-training evaluation of the SAVED BEST CHECKPOINT.
# Reloads the checkpoint and re-runs validation inference. This trains nothing,
# modifies nothing in src/, and never builds a test DataLoader.
# ---------------------------------------------------------------------------
import json
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt

from solarguard.data.datasets import build_train_val_dataloaders
from solarguard.models.baseline_cnn import BaselineCNN, count_parameters
from solarguard.training.checkpoint import load_checkpoint
from solarguard.training.engine import evaluate, resolve_device
from solarguard.training.losses import build_loss, class_weights_from_train_split

REPO_ROOT = Path("/content/Solar_Guard")
DATA_ROOT = REPO_ROOT / "data" / "candidates" / "PV_Panel_Defect_Dataset"
SPLITS_DIR = REPO_ROOT / "data" / "splits"
CHECKPOINT_PATH = REPO_ROOT / "experiments" / "baseline_cnn" / "colab_run_20260818_170845" / "checkpoint_best.pt"

EXPECTED = {
    "epoch": 23,
    "best_metric_name": "macro_f1",
    "best_metric_value": 0.7534464572768266,
}

if not CHECKPOINT_PATH.exists():
    raise FileNotFoundError(f"checkpoint not found: {CHECKPOINT_PATH}")

class_to_idx = json.loads((SPLITS_DIR / "class_mapping.json").read_text())
class_names = sorted(class_to_idx, key=class_to_idx.get)
device = resolve_device("cuda")

# --- 1. reconstruct the exact architecture, then load the checkpoint into it -------
model = BaselineCNN(num_classes=len(class_names))
ckpt = load_checkpoint(CHECKPOINT_PATH, model, map_location=device)
model.to(device)
print(f"architecture : BaselineCNN(num_classes={len(class_names)})")
print(f"parameters   : {count_parameters(model):,}")
print(f"checkpoint   : {CHECKPOINT_PATH}")

# --- 2. confirm checkpoint metadata ------------------------------------------------
print("\n=== checkpoint metadata ===")
for key, expected in EXPECTED.items():
    actual = ckpt.get(key)
    if isinstance(expected, float) and isinstance(actual, float):
        ok = abs(actual - expected) < 1e-12
    else:
        ok = actual == expected
    print(f"  {key:18s} = {actual!r}")
    print(f"  {'':18s}   expected {expected!r} -> {'MATCH' if ok else 'MISMATCH'}")
print(f"  {'seed':18s} = {ckpt.get('seed')!r}")
print(f"  {'class_mapping':18s} = {ckpt.get('class_mapping')}")

# --- 3. VALIDATION split only, using the existing preprocessing config --------------
# build_train_val_dataloaders has no test code path at all (see src/solarguard/data/
# datasets.py) -- the test split cannot be loaded from here even by mistake.
_, val_loader = build_train_val_dataloaders(
    SPLITS_DIR, DATA_ROOT, REPO_ROOT / "configs" / "preprocessing.yaml",
    batch_size=32, num_workers=0, seed=42,
)
print(f"\nvalidation samples : {len(val_loader.dataset)}   (test set NOT loaded)")

# --- 4. inference ------------------------------------------------------------------
# evaluate() calls model.eval() and is decorated @torch.no_grad()
# (src/solarguard/training/engine.py). Both are verified below, not assumed.
class_weights = class_weights_from_train_split(SPLITS_DIR).to(device)
loss_fn = build_loss(class_weights)
results = evaluate(model, val_loader, loss_fn, device, class_names)

assert model.training is False, "model must be in eval mode"
assert all(p.grad is None for p in model.parameters()), "no gradients may exist after a no_grad pass"
print("verified: model.eval() active, no gradients produced")

# --- 5. metrics --------------------------------------------------------------------
per_class = results["per_class"]
macro_precision = float(np.mean([m["precision"] for m in per_class.values()]))
macro_recall = float(np.mean([m["recall"] for m in per_class.values()]))
n_val = sum(m["support"] for m in per_class.values())

print("\n=== validation metrics (reloaded best checkpoint) ===")
print(f"  samples         : {n_val}")
print(f"  accuracy        : {results['accuracy']:.4f}")
print(f"  macro precision : {macro_precision:.4f}")
print(f"  macro recall    : {macro_recall:.4f}")
print(f"  macro F1        : {results['macro_f1']:.4f}")
print(f"  weighted F1     : {results['weighted_f1']:.4f}")
print(f"  val loss        : {results['val_loss']:.4f}")

# --- 6. does the reloaded model reproduce the value recorded at checkpoint time? ----
delta = abs(results["macro_f1"] - EXPECTED["best_metric_value"])
print(f"\nrecorded  macro_f1 : {EXPECTED['best_metric_value']:.16f}")
print(f"recomputed macro_f1 : {results['macro_f1']:.16f}")
print(f"difference          : {delta:.3e} -> {'consistent' if delta < 1e-6 else 'DIVERGED, investigate'}")

# --- 7. classification report ------------------------------------------------------
print("\n=== classification report ===")
print(f"{'class':<20}{'precision':>11}{'recall':>9}{'f1-score':>10}{'support':>9}")
for name in class_names:
    m = per_class[name]
    print(f"{name:<20}{m['precision']:>11.3f}{m['recall']:>9.3f}{m['f1']:>10.3f}{m['support']:>9d}")
print()
print(f"{'accuracy':<20}{'':>11}{'':>9}{results['accuracy']:>10.3f}{n_val:>9d}")
print(f"{'macro avg':<20}{macro_precision:>11.3f}{macro_recall:>9.3f}{results['macro_f1']:>10.3f}{n_val:>9d}")
print(f"{'weighted avg':<20}{'':>11}{'':>9}{results['weighted_f1']:>10.3f}{n_val:>9d}")

# --- 8. confusion matrix -----------------------------------------------------------
cm = np.array(results["confusion_matrix"])
fig, ax = plt.subplots(figsize=(7, 6))
im = ax.imshow(cm, cmap="Blues")
ax.set_xticks(range(len(class_names)))
ax.set_xticklabels(class_names, rotation=45, ha="right")
ax.set_yticks(range(len(class_names)))
ax.set_yticklabels(class_names)
ax.set_xlabel("Predicted")
ax.set_ylabel("True")
ax.set_title(f"Validation Confusion Matrix - best checkpoint (epoch {ckpt.get('epoch')})")
for i in range(len(class_names)):
    for j in range(len(class_names)):
        ax.text(j, i, str(cm[i, j]), ha="center", va="center",
                color="white" if cm[i, j] > cm.max() / 2 else "black")
fig.colorbar(im)
fig.tight_layout()
plt.show()

print("\nTest set was NOT loaded or evaluated in this cell.")